[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Maurilio97-P/6d-pose-vision-workshop/blob/main/part_4_classical_pose/10_pose_with_chessboard.ipynb)

# Notebook 10 — Pose Estimation with a Chessboard
### 6D Pose Vision Workshop · Part 4: Classical Pose Estimation

**Estimated time:** 60 minutes  
**Dependencies:** `opencv-contrib-python`, `numpy`, `matplotlib`

> **Grounded in:** video note 2 (*OpenCV Python Pose Estimation for Objects*)

This is the **first full end-to-end pipeline** notebook. By the end you will have a working system that:
1. Loads camera calibration (K, dist)
2. Detects a chessboard in an image
3. Runs `solvePnP` to get 6D pose
4. Draws 3D coordinate axes on the board
5. Draws a 3D cube sitting on the board

---

## Recommended Watch

> Same video as NB 09 — it covers the full solvePnP pipeline built in this notebook. Skip if you already watched it.

| Title | Link | Duration |
|---|---|---|
| **OpenCV Python Pose Estimation for Objects (Algorithm and Code)** | [▶ Watch](https://www.youtube.com/watch?v=bs81DNsMrnM) | ~20 min |

---

In [ ]:
import sys
IN_COLAB = 'google.colab' in sys.modules
if IN_COLAB:
    !pip install opencv-contrib-python numpy matplotlib -q
    print("Running in Google Colab")
else:
    print("Running locally — make sure your venv is activated")

import cv2
import numpy as np
import matplotlib.pyplot as plt
import os

print(f"OpenCV: {cv2.__version__}")

def show_images(pairs, figsize_per=(6, 5)):
    n = len(pairs)
    fig, axes = plt.subplots(1, n, figsize=(figsize_per[0]*n, figsize_per[1]))
    if n == 1: axes = [axes]
    for ax, (img, title) in zip(axes, pairs):
        ax.imshow(img[:,:,::-1] if img.ndim==3 else img, cmap=None if img.ndim==3 else 'gray')
        ax.set_title(title, fontsize=9); ax.axis('off')
    plt.tight_layout(); plt.show()

## Learning Objectives

By the end of this notebook you will:

- Build a **complete pipeline**: calibration → detection → pose → visualization
- Draw **3D coordinate axes** on a detected object using the estimated pose
- Draw a **3D cube** that appears to sit on the chessboard surface
- Process **multiple images** and track how pose changes across views
- Know how to adapt this pipeline to **any flat object** (ArUco markers, custom targets)

---

## 1. Pipeline Overview

```
┌───────────────────────────────────────────────────────┐
│              FULL POSE ESTIMATION PIPELINE            │
│                                                       │
│  [Calibration data]  →  K, dist                       │
│         +                                             │
│  [Input frame]       →  grayscale                     │
│         +                                             │
│  [Object model]      →  3D corner positions (objpts)  │
│                                ↓                      │
│                  findChessboardCorners                │
│                        ↓                             │
│                  cornerSubPix                         │
│                        ↓                             │
│                  solvePnP(objpts, imgpts, K, dist)    │
│                        ↓                             │
│                  rvec, tvec  ← 6D pose               │
│                        ↓                             │
│                  projectPoints(3D shapes)             │
│                        ↓                             │
│                  Draw axes / cube on frame            │
└───────────────────────────────────────────────────────┘
```

In [ ]:
# Step 1: Load calibration

calib_path = '../assets/calibration/camera_calibration.npz'

if os.path.exists(calib_path):
    data = np.load(calib_path)
    K    = data['K']
    dist = data['dist']
    print(f"Loaded calibration from notebook 07")
else:
    # Sensible defaults if notebook 07 wasn't run
    K    = np.array([[720, 0, 320], [0, 720, 240], [0, 0, 1]], dtype=np.float64)
    dist = np.array([-0.3, 0.1, 0.001, 0.002, -0.05])
    print("Using default K — run notebook 07 for real calibration")

print(f"K:\n{K}")
print(f"dist: {dist.flatten()}")

In [ ]:
# Step 2: Define chessboard geometry

CHESSBOARD_SIZE = (9, 6)   # (cols, rows) inner corners
SQUARE_SIZE     = 0.025    # 2.5 cm per square (adjust to your actual board)

# Object points: inner corners at Z=0, spaced SQUARE_SIZE apart
objp = np.zeros((CHESSBOARD_SIZE[0] * CHESSBOARD_SIZE[1], 3), np.float32)
objp[:, :2] = np.mgrid[0:CHESSBOARD_SIZE[0], 0:CHESSBOARD_SIZE[1]].T.reshape(-1, 2)
objp *= SQUARE_SIZE

print(f"Object point grid: {CHESSBOARD_SIZE[0]} × {CHESSBOARD_SIZE[1]} = {len(objp)} corners")
print(f"Board dimensions: {CHESSBOARD_SIZE[0]*SQUARE_SIZE*100:.1f}cm × {CHESSBOARD_SIZE[1]*SQUARE_SIZE*100:.1f}cm")
print(f"Corner [0]: {objp[0]}  — top-left")
print(f"Corner [8]: {objp[8]}  — top-right")
print(f"Corner [-1]: {objp[-1]} — bottom-right")

# Termination criteria for cornerSubPix
CRITERIA = (cv2.TERM_CRITERIA_EPS + cv2.TERM_CRITERIA_MAX_ITER, 30, 0.001)

In [ ]:
# Step 3: Define visualization functions

def draw_axes(img, rvec, tvec, K, dist, axis_length=None):
    """
    Draw XYZ coordinate axes on a detected object.
    X=red, Y=green, Z=blue. Axis length = 1 chessboard square.
    """
    if axis_length is None:
        axis_length = SQUARE_SIZE * 3  # 3 squares long
    
    axis_3d = np.float32([
        [0, 0, 0],                    # origin
        [axis_length, 0, 0],          # X axis
        [0, axis_length, 0],          # Y axis
        [0, 0, -axis_length],         # Z axis (points out of board toward camera)
    ])
    
    pts, _ = cv2.projectPoints(axis_3d, rvec, tvec, K, dist)
    pts = pts.reshape(-1, 2).astype(int)
    
    origin = tuple(pts[0])
    cv2.arrowedLine(img, origin, tuple(pts[1]), (0, 0, 255), 3, tipLength=0.2)  # X red
    cv2.arrowedLine(img, origin, tuple(pts[2]), (0, 255, 0), 3, tipLength=0.2)  # Y green
    cv2.arrowedLine(img, origin, tuple(pts[3]), (255, 0, 0), 3, tipLength=0.2)  # Z blue
    
    # Labels
    cv2.putText(img, 'X', tuple(pts[1]), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,0,255), 2)
    cv2.putText(img, 'Y', tuple(pts[2]), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)
    cv2.putText(img, 'Z', tuple(pts[3]), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (255,0,0), 2)
    
    return img


def draw_cube(img, rvec, tvec, K, dist, cube_size=None, offset_x=0, offset_y=0):
    """
    Draw a 3D cube sitting on the chessboard surface at the given offset.
    cube_size: side length in meters. Default = 3 chessboard squares.
    """
    if cube_size is None:
        cube_size = SQUARE_SIZE * 3
    
    s = cube_size
    ox, oy = offset_x, offset_y
    
    # 8 corners: bottom face (Z=0) and top face (Z=-s, pointing toward camera)
    cube_3d = np.float32([
        [ox,   oy,   0], [ox+s, oy,   0], [ox+s, oy+s, 0], [ox,   oy+s, 0],  # bottom
        [ox,   oy,  -s], [ox+s, oy,  -s], [ox+s, oy+s,-s], [ox,   oy+s,-s],  # top
    ])
    
    pts, _ = cv2.projectPoints(cube_3d, rvec, tvec, K, dist)
    pts = pts.reshape(-1, 2).astype(int)
    
    def draw_face(p_indices, color, thickness=2):
        pts_face = [tuple(pts[i]) for i in p_indices]
        for i in range(len(pts_face)):
            cv2.line(img, pts_face[i], pts_face[(i+1) % len(pts_face)], color, thickness)
    
    draw_face([0,1,2,3], (0, 200, 0), 2)      # bottom face — green
    draw_face([4,5,6,7], (200, 100, 0), 2)    # top face — blue-ish
    for i in range(4):                          # vertical edges — white
        cv2.line(img, tuple(pts[i]), tuple(pts[i+4]), (200, 200, 200), 1)
    
    return img


def annotate_pose(img, rvec, tvec):
    """Add pose info text overlay."""
    tx, ty, tz = tvec.flatten()
    dist_m = np.linalg.norm(tvec)
    angle_deg = np.degrees(np.linalg.norm(rvec))
    
    lines = [
        f"X: {tx:.3f}m  Y: {ty:.3f}m  Z: {tz:.3f}m",
        f"dist: {dist_m:.3f}m   angle: {angle_deg:.1f}deg",
    ]
    for i, line in enumerate(lines):
        cv2.putText(img, line, (10, 25 + i*28),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.65, (255, 255, 0), 2)
    return img

print("Drawing functions ready: draw_axes(), draw_cube(), annotate_pose()")

In [ ]:
# Step 4: Full pipeline — process all calibration images as test inputs

import glob

def process_frame(img_bgr, K, dist, chessboard_size, objp, criteria):
    """
    Full pipeline: detect chessboard → solvePnP → draw axes + cube.
    Returns: (annotated_img, rvec, tvec, success)
    """
    gray = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY)
    result = img_bgr.copy()
    
    # --- Detection ---
    ret, corners = cv2.findChessboardCorners(gray, chessboard_size, None)
    
    if not ret:
        cv2.putText(result, 'Chessboard NOT detected', (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
        return result, None, None, False
    
    # Refine corners
    corners_refined = cv2.cornerSubPix(gray, corners, (11, 11), (-1, -1), criteria)
    
    # Draw detected corners
    cv2.drawChessboardCorners(result, chessboard_size, corners_refined, ret)
    
    # --- Pose estimation ---
    ret_pnp, rvec, tvec = cv2.solvePnP(
        objp, corners_refined, K, dist,
        flags=cv2.SOLVEPNP_ITERATIVE
    )
    
    if not ret_pnp:
        cv2.putText(result, 'solvePnP FAILED', (10, 30),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.8, (0, 0, 255), 2)
        return result, None, None, False
    
    # --- Compute reprojection error ---
    projected, _ = cv2.projectPoints(objp, rvec, tvec, K, dist)
    reproj_err = np.mean(np.linalg.norm(
        corners_refined.reshape(-1, 2) - projected.reshape(-1, 2), axis=1))
    
    # --- Visualization ---
    draw_axes(result, rvec, tvec, K, dist)
    draw_cube(result, rvec, tvec, K, dist,
              cube_size=SQUARE_SIZE*3,
              offset_x=SQUARE_SIZE*2,
              offset_y=SQUARE_SIZE*1)
    annotate_pose(result, rvec, tvec)
    cv2.putText(result, f'reproj: {reproj_err:.3f}px', (10, 90),
                cv2.FONT_HERSHEY_SIMPLEX, 0.55, (255,255,255), 1)
    
    return result, rvec, tvec, True


# Process all available calibration images
image_paths = sorted(glob.glob('../assets/calibration/chessboard_images/calib_*.jpg'))

results_to_show = []
pose_data = []

for i, path in enumerate(image_paths):
    img = cv2.imread(path)
    if img is None:
        continue
    
    annotated, rvec, tvec, success = process_frame(
        img, K, dist, CHESSBOARD_SIZE, objp, CRITERIA)
    
    if success:
        pose_data.append({'image': i, 'rvec': rvec.flatten(), 'tvec': tvec.flatten()})
        if i < 4:  # show first 4
            results_to_show.append((annotated, f'Image {i}\nZ={tvec.flatten()[2]:.2f}m'))
    
print(f"Processed {len(pose_data)}/{len(image_paths)} images successfully")

if results_to_show:
    show_images(results_to_show[:3], figsize_per=(6, 5))

In [ ]:
# Visualize how pose changes across the calibration image set

if pose_data:
    indices = [d['image'] for d in pose_data]
    tvecs = np.array([d['tvec'] for d in pose_data])
    rvecs = np.array([d['rvec'] for d in pose_data])
    angles = np.degrees([np.linalg.norm(r) for r in rvecs])
    distances = np.linalg.norm(tvecs, axis=1)
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 8))
    
    # Translation components
    for i, (label, color) in enumerate([('X (right)', 'red'), ('Y (down)', 'green'), ('Z (depth)', 'blue')]):
        axes[0,0].plot(indices, tvecs[:,i]*100, label=label, color=color, marker='o')
    axes[0,0].set_title('Translation per image (cm)')
    axes[0,0].set_xlabel('Image index'); axes[0,0].set_ylabel('cm')
    axes[0,0].legend(); axes[0,0].grid(True, alpha=0.3)
    
    # Distance from camera
    axes[0,1].plot(indices, distances*100, 'purple', marker='s')
    axes[0,1].set_title('Distance from camera (cm)')
    axes[0,1].set_xlabel('Image index'); axes[0,1].set_ylabel('cm')
    axes[0,1].grid(True, alpha=0.3)
    
    # Rotation angle
    axes[1,0].plot(indices, angles, 'orange', marker='D')
    axes[1,0].set_title('Rotation angle (degrees)')
    axes[1,0].set_xlabel('Image index'); axes[1,0].set_ylabel('degrees')
    axes[1,0].grid(True, alpha=0.3)
    
    # X-Y plane trajectory
    axes[1,1].scatter(tvecs[:,0]*100, tvecs[:,1]*100, c=indices, cmap='viridis', s=80)
    for i, idx in enumerate(indices):
        axes[1,1].annotate(str(idx), (tvecs[i,0]*100, tvecs[i,1]*100), fontsize=7)
    axes[1,1].set_title('X-Y position of board center (cm)')
    axes[1,1].set_xlabel('X (right, cm)'); axes[1,1].set_ylabel('Y (down, cm)')
    axes[1,1].grid(True, alpha=0.3); axes[1,1].invert_yaxis()
    
    plt.tight_layout()
    plt.show()
    print("Each image was taken at a different angle/distance → pose varies accordingly")

## 2. Real-Time Version (Webcam)

The full real-time pipeline is shown below as a script template. Run it as a `.py` file with a chessboard visible to your webcam:

## Running the real-time chessboard pose estimation script

The script `scripts/pose/chessboard_pose_estimation.py` opens your webcam and continuously estimates the full 6D pose (position + orientation) of a printed chessboard. It draws 3D coordinate axes at the board origin and optionally projects a 3D wireframe cube sitting on the board.

---

### Before you run

You need your camera calibration file first. Run NB07 to generate `assets/calibration/camera_calibration.npz`. Without it the script uses a rough fallback and the distance numbers will not be accurate in real-world units.

Also print the chessboard from NB07 (`assets/calibration/chessboard_9x6.png`) and measure the physical side length of one square with a ruler before running.

Then set up your virtual environment if you haven't already:

```bash
# 1. Create the venv (only once):
python -m venv venv

# 2. Activate it (every time you open a new terminal):
#    Windows:
venv\Scripts\activate
#    macOS / Linux:
source venv/bin/activate

# 3. Install dependencies (only once, after activating):
pip install -r requirements.txt
```

You should see `(venv)` at the start of your terminal prompt.  
If you don't, the venv is not active and the script will fail with `ModuleNotFoundError: No module named 'cv2'`.

---

### What the script does, step by step

1. Loads camera intrinsics (K, dist) from the calibration file.
2. Builds the grid of 3D object points (the known corner positions of the chessboard in real-world coordinates, in metres).
3. Opens the webcam and starts the main loop.
4. Each frame: converts to grayscale and calls `findChessboardCorners` to locate the inner corner grid.
5. When the board is found, refines the corners to sub-pixel accuracy with `cornerSubPix`, then calls `solvePnP` to compute the 6D pose (rvec + tvec).
6. Draws colour-coded 3D axes at the board origin: **Red = X**, **Green = Y**, **Blue = Z** (pointing up from the board surface).
7. If the cube overlay is enabled, projects a 3D wireframe cube sitting on the board at squares 2–4.
8. Overlays the board-to-camera distance Z and the lateral/vertical offsets X, Y in centimetres.
9. If the board is not visible, shows `Hold chessboard in view...` as a prompt.

---

### How to run

```bash
# Default: 9×6 board, 2.5 cm squares, camera 0
python scripts/pose/chessboard_pose_estimation.py

# Specify board dimensions and square size
python scripts/pose/chessboard_pose_estimation.py --cols 9 --rows 6 --square 0.025

# Specify calibration file explicitly
python scripts/pose/chessboard_pose_estimation.py --calib assets/calibration/camera_calibration.npz
```

Key arguments:
- `--calib` — Path to the `.npz` calibration file from NB07.
- `--cols` — Inner corner columns (default: `9`; one less than the number of squares across).
- `--rows` — Inner corner rows (default: `6`; one less than the number of squares down).
- `--square` — Physical side length of one square **in metres** (default: `0.025` = 2.5 cm). Measure with a ruler.
- `--camera` — Camera index (default: `0`).
- `--width` / `--height` — Capture resolution (default: `1280 × 720`).

---

### Controls

| Key | Action |
|-----|--------|
| `Q` or `Esc` | Quit the live window |
| `C` | Toggle the 3D wireframe cube overlay on / off |

---

### What output to expect

A window opens showing your webcam feed. When you hold the printed chessboard in front of the camera:
- All inner corners are highlighted in colour.
- Three coloured arrows grow from the top-left corner of the board: Red (X), Green (Y), Blue (Z).
- If the cube is on, a green wireframe cube sits on the board.
- The terminal overlay shows `Z=...cm  X=...  Y=...` in the top-left corner.

When the board is not visible, the text `Hold chessboard in view...` is shown in orange.  
No files are saved. For best results: use real calibration data from NB07, set `--square` to your actual measured square size, mount the board flat on a hard surface, and use even diffuse lighting with no glare on the printed squares.

## Recap — Full Pipeline Summary

```
ONCE (startup):
  1. Load K, dist from calibration
  2. Build objp (3D grid in world frame)

PER FRAME:
  3. findChessboardCorners → ret, corners
  4. if ret: cornerSubPix → corners_refined
  5. solvePnP(objp, corners_refined, K, dist) → rvec, tvec
  6. projectPoints(3D_shapes, rvec, tvec, K, dist) → 2D points
  7. Draw projected shapes on frame
  8. Display / publish pose to robot
```

| Step | Input | Output |
|---|---|---|
| `findChessboardCorners` | gray frame | 2D corner pixel locations |
| `cornerSubPix` | coarse corners | subpixel-accurate corners |
| `solvePnP` | 3D objp + 2D corners + K | rvec, tvec |
| `projectPoints` | 3D shapes + rvec/tvec + K | 2D pixel locations |
| `drawAxes/Cube` | frame + 2D shapes | annotated frame |

---

**Next:** `11_aruco_theory.ipynb` — replace the chessboard with an ArUco marker. The pipeline is identical, but detection is instant, robust, and works with a single marker instead of a large chessboard.

## Exercises

In [ ]:
# ============================================================
# EXERCISE 1: Draw a custom shape on the chessboard
# ============================================================
# Modify the pipeline to draw a pyramid (4 base points + 1 apex) instead of a cube.
# The pyramid should sit on the board with:
#   - Square base: 3x3 chessboard squares
#   - Height: 3 chessboard squares (pointing away from board)
# Draw the 4 base-to-apex edges and the base square.

# YOUR CODE HERE


# ============================================================
# SOLUTION (uncomment to reveal)
# ============================================================
# def draw_pyramid(img, rvec, tvec, K, dist, base_size=None, height=None, offset_x=0, offset_y=0):
#     if base_size is None: base_size = SQUARE_SIZE * 3
#     if height is None: height = SQUARE_SIZE * 3
#     s, h = base_size, height
#     ox, oy = offset_x, offset_y
#
#     pyramid_3d = np.float32([
#         [ox,   oy,   0],     # base corners
#         [ox+s, oy,   0],
#         [ox+s, oy+s, 0],
#         [ox,   oy+s, 0],
#         [ox+s/2, oy+s/2, -h],  # apex
#     ])
#     pts, _ = cv2.projectPoints(pyramid_3d, rvec, tvec, K, dist)
#     pts = pts.reshape(-1, 2).astype(int)
#     base = pts[:4]; apex = tuple(pts[4])
#
#     # Draw base
#     for i in range(4):
#         cv2.line(img, tuple(base[i]), tuple(base[(i+1)%4]), (0, 200, 100), 2)
#     # Draw apex edges
#     for i in range(4):
#         cv2.line(img, tuple(base[i]), apex, (200, 150, 0), 2)
#     return img
#
# # Test on one image
# img = cv2.imread(image_paths[0])
# gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
# ret, corners = cv2.findChessboardCorners(gray, CHESSBOARD_SIZE, None)
# if ret:
#     cr = cv2.cornerSubPix(gray, corners, (11,11), (-1,-1), CRITERIA)
#     _, rv, tv = cv2.solvePnP(objp, cr, K, dist)
#     out = img.copy()
#     draw_pyramid(out, rv, tv, K, dist, offset_x=SQUARE_SIZE, offset_y=SQUARE_SIZE)
#     show_images([(out, 'Pyramid on chessboard')])

In [ ]:
# ============================================================
# EXERCISE 2: Pose comparison across methods
# ============================================================
# Take one calibration image. Run solvePnP with these methods:
#   - SOLVEPNP_ITERATIVE
#   - SOLVEPNP_EPNP
#   - SOLVEPNP_AP3P (needs exactly 4 points — use first 4 objp/imgp)
#
# For each: print rvec, tvec, and reprojection error.
# Which method gives the lowest reprojection error on this image?

# YOUR CODE HERE


# ============================================================
# SOLUTION (uncomment to reveal)
# ============================================================
# img = cv2.imread(image_paths[0])
# gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
# ret, corners = cv2.findChessboardCorners(gray, CHESSBOARD_SIZE, None)
# if not ret:
#     print("No corners found"); exit()
# cr = cv2.cornerSubPix(gray, corners, (11,11), (-1,-1), CRITERIA)
#
# methods = [
#     ('ITERATIVE', cv2.SOLVEPNP_ITERATIVE, objp,    cr),
#     ('EPNP',      cv2.SOLVEPNP_EPNP,      objp,    cr),
#     ('AP3P',      cv2.SOLVEPNP_AP3P,      objp[:4],cr[:4]),
# ]
# print(f"{'Method':<15} {'rvec norm (rad)':<20} {'tvec Z (m)':<15} {'reproj error (px)'}")
# print("-" * 65)
# for name, flag, op, ip in methods:
#     r, rv, tv = cv2.solvePnP(op, ip, K, dist, flags=flag)
#     proj, _ = cv2.projectPoints(op, rv, tv, K, dist)
#     err = np.mean(np.linalg.norm(ip.reshape(-1,2)-proj.reshape(-1,2), axis=1))
#     print(f"{name:<15} {np.linalg.norm(rv):<20.4f} {tv.flatten()[2]:<15.4f} {err:.4f}")